# 01 · Business problem, data understanding and data quality
**Steps 1–4 of the project.**

## Step 1: Business problem
An e-commerce marketplace stores orders, items, payments, reviews, customers, sellers and products in 9 related tables. The goal is to turn this raw data into insights about sales, customers, sellers, delivery and satisfaction.

**Questions this project answers**
1. How are revenue and orders changing month by month?
2. Which categories, products and states drive revenue?
3. How many customers come back for a second order?
4. How concentrated is revenue among sellers?
5. How long does delivery take, and how often is it late?
6. Do late deliveries lead to lower review scores? *(t-test)*
7. Does order value differ by product category? *(ANOVA)*
8. Is payment method associated with order status? *(chi-square)*
9. Which states and sellers have the worst delivery and ratings?
10. What should the business do first to improve?

In [ ]:
import sys
sys.path.append("..")  # so `src` can be imported from the notebooks folder

import pandas as pd
pd.set_option("display.max_columns", 50)

from src.config import RAW_DIR
from src.data_loader import load_raw_tables, describe_tables, build_data_dictionary
from src.data_quality import run_quality_report

## Step 2: Dataset and ER diagram
| Table | Primary key | Foreign keys |
| --- | --- | --- |
| customers | customer_id | — |
| orders | order_id | customer_id → customers |
| order_items | order_id + order_item_id | order_id → orders, product_id → products, seller_id → sellers |
| payments | order_id + payment_sequential | order_id → orders |
| reviews | review_id | order_id → orders |
| products | product_id | product_category_name → category_translation |
| sellers | seller_id | — |
| category_translation | product_category_name | — |
| geolocation | (none in raw file) | linked by zip code prefix |

Key point: **one order can have many items and many payments**, so these must be aggregated to order level before joining.

## Step 3: Load the raw data

In [ ]:
raw = load_raw_tables(RAW_DIR)
describe_tables(raw)

In [ ]:
for name, df in raw.items():
    print(f"\n=== {name} ===")
    display(df.head(3))

### Data dictionary

In [ ]:
build_data_dictionary(raw)

## Step 4: Data quality analysis

In [ ]:
report = run_quality_report(raw)
for section, df in report.items():
    print(f"\n### {section}")
    display(df)

### What the quality checks mean for cleaning
Write your own notes here after running the cells, for example:
- Which columns have missing values, and why (e.g. delivery date is empty for orders not yet delivered).
- Which tables have duplicate keys (e.g. more than one review per order).
- Which values break business rules (e.g. `not_defined` payment type).
- Outliers in price are real expensive products, so they are **kept**, not removed.